# Qwen3-8B H4rmony R1-only LoRA SFT

This notebook fine-tunes `Qwen/Qwen3-8B` on one prompt → R1 example per H4rmony `PromptID`. Here **R1 means the most environmentally aligned answer**, not the DeepSeek-R1 model. The reusable data, tokenization, training, evaluation, and artifact-validation code lives in `scripts/harmony_sft/`; the notebook is only the Colab entry point.

Use an A100 (40 GB or larger). Checkpoints, the final LoRA adapter, the exact selected dataset snapshot, training metrics, and the matched base-versus-SFT ecological evaluation are written directly to Google Drive. The final adapter is small and must be loaded on top of the pinned Qwen3-8B base model recorded in the run metadata.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/shengweiming/value-misalignment.git"
REPO_DIR = Path("/content/value-misalignment")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)

In [ ]:
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from google.colab import drive
drive.mount("/content/drive")

import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then retry."
assert torch.cuda.is_bf16_supported(), "This workflow requires a BF16-capable GPU."
gpu = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu.total_memory / 2**30
assert gpu_memory_gib >= 38, "Select an A100 40 GB (or larger) runtime for this configuration."
print(f"GPU: {gpu.name} ({gpu_memory_gib:.1f} GiB)")

The default intervention is BF16 LoRA with rank 16, alpha 32, dropout 0.05, and three epochs. Loss is applied only to the R1 assistant answer (plus its end token), with Qwen thinking strictly disabled. Change `RUN_NAME` only when you want a stable human-readable folder name; otherwise a UTC timestamp prevents accidental overwrites.

In [ ]:
from scripts.harmony_sft import SFTConfig

DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/value-misalignment/harmony_r1_qwen3_8b")
RUN_NAME = None

CONFIG = SFTConfig(
    output_root=DRIVE_OUTPUT_ROOT,
    run_name=RUN_NAME,
    base_model="Qwen/Qwen3-8B",
    dataset_id="neovalle/H4rmony",
    max_length=1024,
    num_train_epochs=3,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    lora_rank=16,
    lora_alpha=32,
    lora_dropout=0.05,
    eval_batch_size=4,
    seed=42,
)
CONFIG

In [ ]:
from scripts.harmony_sft import run_harmony_r1_sft

artifacts = run_harmony_r1_sft(CONFIG)
print("Completed Drive run:", artifacts.run_dir)

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

complete = json.loads(artifacts.complete_marker_path.read_text())
train_metrics = json.loads(artifacts.train_metrics_path.read_text())
assert complete["status"] == "complete"
assert artifacts.final_adapter_dir.is_dir()
display(pd.DataFrame([train_metrics]))
display(pd.read_csv(artifacts.thresholds_path))
display(Image(filename=str(artifacts.plot_path)))
print("Final adapter:", artifacts.final_adapter_dir)
print("All checkpoints and results:", artifacts.run_dir)

A `COMPLETE.json` marker is written only after the final adapter and required result files have been verified and hashed. If the run fails, inspect `FAILED.json` in the new Drive run folder. The four ecological evaluation templates remain an exploratory screen rather than a confirmatory generalization result.